In [ ]:
!pip install sentence-transformers pandas tqdm python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.9 MB/s eta 0:00:00


In [ ]:
import json
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util

In [ ]:
import os

for root, dirs, files in os.walk('/content'):
    for file in files:
        print(os.path.join(root, file))

/content/.config/.last_opt_in_prompt.yaml
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/.config/active_config
/content/.config/default_configs.db
/content/.config/gce
/content/.config/config_sentinel
/content/.config/.last_update_check.json
/content/.config/.last_survey_prompt.yaml
/content/.config/configurations/config_default
/content/.config/logs/2026.06.04/13.32.39.344962.log
/content/.config/logs/2026.06.04/13.32.18.735754.log
/content/.config/logs/2026.06.04/13.32.02.654775.log
/content/.config/logs/2026.06.04/13.32.21.210570.log
/content/.config/logs/2026.06.04/13.32.38.346437.log
/content/.config/logs/2026.06.04/13.31.42.499627.log
/content/sample_data/anscombe.json
/content/sample_data/README.md
/content/sample_data/california_housing_test.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/mnist_test.csv


In [4]:
import os

print(os.listdir('/content'))

['.config', '[PUB] India_runs_data_and_ai_challenge.zip', 'sample_data']


In [5]:
candidates_file = "/content/candidates.jsonl"
job_file = "/content/job_description.docx"

print(candidates_file)
print(job_file)

/content/candidates.jsonl
/content/job_description.docx


In [6]:
from docx import Document

doc = Document(job_file)

job_text = ""

for para in doc.paragraphs:
    job_text += para.text + "\n"

print(job_text[:1000])

Job Description: Senior AI Engineer — Founding Team
Company: Redrob AI (Series A AI-native talent intelligence platform)
Location: Pune/Noida, India (Hybrid — flexible cadence) | Open to relocation candidates from Tier-1 Indian cities
Employment Type: Full-time
Experience Required: 5–9 years (see "what we mean by this" below)

Let's be honest about this role
We're going to write this JD differently from most. We're a Series A company that just raised our round and we're building a new AI Engineering org from scratch. This is the kind of role where the JD changes every six months because the company changes every six months. So instead of pretending we have a fixed checklist, we're going to tell you what we actually need and what we've gotten wrong before.
If you've spent your career at Google or Meta and you want a well-scoped role with a defined ladder, this isn't it.
If you've spent your career bouncing between early-stage startups and you want to "just code" without having to think 

In [7]:
import json

with open(candidates_file, "r", encoding="utf-8") as f:
    first_candidate = json.loads(next(f))

print(first_candidate.keys())

dict_keys(['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals'])


In [8]:
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Compute job embedding only ONCE
job_embedding = model.encode(job_text, convert_to_tensor=True)

candidate_scores = []

with open(candidates_file, "r", encoding="utf-8") as f:

    for line in tqdm(f):
        cand = json.loads(line)

        text = ""

        # Profile
        if 'profile' in cand:
            profile = cand['profile']
            text += profile.get('headline', '') + " "
            text += profile.get('summary', '') + " "
            text += profile.get('current_title', '') + " "

        # Skills
        if 'skills' in cand:
            if isinstance(cand['skills'], list):
                text += " ".join(map(str, cand['skills'])) + " "

        # Career History
        if 'career_history' in cand:
            for exp in cand['career_history']:
                text += exp.get('title', '') + " "
                text += exp.get('company', '') + " "
                text += exp.get('description', '') + " "

        # Education
        if 'education' in cand:
            for edu in cand['education']:
                text += edu.get('degree', '') + " "
                text += edu.get('institution', '') + " "

        # Candidate embedding
        cand_embedding = model.encode(text, convert_to_tensor=True)

        score = util.cos_sim(job_embedding, cand_embedding).item()

        candidate_scores.append({
            "candidate_id": cand["candidate_id"],
            "score": score
        })

# Convert to DataFrame
df = pd.DataFrame(candidate_scores)

# Normalize scores between 0 and 100
df["score"] = ((df["score"] - df["score"].min()) /
               (df["score"].max() - df["score"].min())) * 100

# Add tiny noise to avoid exact ties
df["score"] += np.random.uniform(0, 0.1, len(df))

# Sort descending
df = df.sort_values("score", ascending=False)

# Assign ranks
df["rank"] = range(1, len(df) + 1)

# Round scores
df["score"] = df["score"].round(4)

# Show top 20
print(df.head(20))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

100000it [2:37:00, 10.61it/s]


       candidate_id     score  rank
39976  CAND_0039977  100.0807     1
13263  CAND_0013264   96.9260     2
78120  CAND_0078121   96.6836     3
53052  CAND_0053053   96.2251     4
24857  CAND_0024858   95.6610     5
31651  CAND_0031652   95.5231     6
7308   CAND_0007309   95.4632     7
95946  CAND_0095947   95.4450     8
68511  CAND_0068512   95.4415     9
63656  CAND_0063657   95.2102    10
37002  CAND_0037003   95.1045    11
81471  CAND_0081472   95.0812    12
15132  CAND_0015133   94.5125    13
22041  CAND_0022042   94.4572    14
27854  CAND_0027855   94.4205    15
71907  CAND_0071908   94.4015    16
13809  CAND_0013810   94.2646    17
41894  CAND_0041895   94.2132    18
39715  CAND_0039716   94.0907    19
55256  CAND_0055257   94.0079    20


In [9]:
# Save all ranked candidates
df.to_csv("shortlisted_candidates.csv", index=False)

print("File saved successfully!")

File saved successfully!


In [10]:
print(df.shape)
print(df.head())

(100000, 3)
       candidate_id     score  rank
39976  CAND_0039977  100.0807     1
13263  CAND_0013264   96.9260     2
78120  CAND_0078121   96.6836     3
53052  CAND_0053053   96.2251     4
24857  CAND_0024858   95.6610     5
